### 슬라이드 분석 모듈

#### 환경 설정

In [67]:
# library import
import os
from dotenv import load_dotenv
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 환경 변수
load_dotenv()
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

#### Structured Output 정의

In [68]:
# Slide 모델 정의
class Slide(BaseModel):
    slide_number: int | None = Field(description="슬라이드 번호. 1부터 시작하는 정수. 구분 불가능할 경우 None")
    script: str = Field(description="해당 슬라이드에 포함된 발표 대본")
    keywords: list[str] = Field(description="해당 슬라이드 대본에 포함된 키워드 목록")
    highlights: list[str] = Field(description="해당 슬라이드 대본에서 강조할 부분 인덱스 목록")

# 전체 결과 모델 정의
class SeperatedSlides(BaseModel):
    status: Literal["success", "fail"] = Field(description="슬라이드 구분 성공 여부")
    slides: list[Slide] = Field(description="슬라이드별 발표 대본 목록")
    terms: list[str] = Field(description="발표 대본에서 추출한 용어 목록")

#### System Prompt 작성

In [69]:
SYSTEM_PROMPT = """
너는 발표 대본을 슬라이드별로 분리하는 전문가야.
입력은 하나의 발표 대본 전체 텍스트야.

# 슬라이드 구분 판단 기준
- "Slide", "슬라이드" 또는 숫자같은 명시적 표기만 슬라이드 구분으로 인정해.
- "첫째", "둘째", "다음으로" 같은 서수/전환 표현은 슬라이드 구분이 아니야.
- 대본 전체에서 이런 명시적 구분이 하나도 없거나, 일부 구간에만 있고 나머지 구간은 구분할 수 없으면 반드시 status="fail"로 처리해. 임의로 슬라이드 번호를 만들어내거나 추측해서 나누지 마.

# keywords 공통 규칙 (success/fail 모두 적용, 개수만 다름)
- keywords는 대본에 있는 단어나 어절을 그대로 사용해. 문장 내용을 바탕으로 요약하거나 의미를 추출하지 마.
- keywords의 순서는 대본에 등장하는 순서대로 담아. (중복 단어는 제거해)
- keywords에는 의미 없는 단어, 조사/접속사 등은 포함하지 마.
- 쉼표(,)로 나열된 단어/구는 서로 의미가 다르더라도 무조건 그 중 하나만 keywords에 담아. 나열된 개수만큼 여러 개의 keyword로 쪼개지 마. (예: "발표 시간, 말하기 속도, pause, 목소리 크기, 시선 처리, filler word" -> 이 중 하나만 담기, 나머지는 keywords에서 제외)

# highlights 공통 규칙
- highlights 필드는 빈 배열로 채워. (추후에 강조할 부분을 표시하기 위해 마련한 필드임)

# terms 공통 규칙 (슬라이드 단위가 아니라 대본 전체를 기준으로 한 번만 추출)
- terms는 STT(음성 인식) 모델이 잘못 인식할 가능성이 있는 고유명사 또는 전문 용어만 추출한 목록이야. keywords와는 추출 기준이 다르니 혼동하지 마.
- 판단 기준은 "중요한 단어"가 아니라 "STT가 학습 데이터에서 충분히 접하지 못했을 가능성이 높은 단어"야. 아래 조건을 모두 만족해야 terms에 포함할 수 있어.
  - 표준국어대사전에 등재되어 있거나 일상 대화에서 흔히 쓰여 이미 대중적으로 굳어진 일반 명사/합성어가 아닐 것. (예: "전기차", "인공지능", "스마트폰", "재활용"은 흔히 쓰이는 단어이므로 제외)
  - 발음이나 표기가 여러 가지로 혼동될 수 있거나, 처음 듣는 사람이 표기를 유추하기 어려운 단어일 것.
- 포함 대상 예시: 특정 인명, 지명(고유 지명), 기업명, 제품명, 서비스명, 브랜드명, 모델명, 논문/이론 명칭, 영문 약어, 신조어, 외래어 음차 표기, 특정 학문/산업 분야의 전문 용어(일반인에게 생소한 수준).
- 제외 대상: 일반적인 단어, 일상적인 표현, 이미 널리 쓰여 국어사전에 등재된 합성어, 단순히 문맥상 중요하기만 한 키워드.
- 대본에 실제로 등장하는 표현만 그대로 추출해. 요약하거나 변형하지 마.
- 중복된 용어는 하나만 남겨.
- terms는 최대 100개까지만 담아. 100개를 초과할 경우 STT 인식 실패 가능성이 더 높은 용어를 우선적으로 남겨.

# 전처리 공통 규칙
- 대본 최상단에 제목이 있을 경우 제거해.

# status="success"인 경우
- 대본 전체가 명시적 구분자를 기준으로 빠짐없이 나뉠 수 있을 때만 success로 판단해.
- slides 배열에 모든 슬라이드를 slide_number, script와 함께 순서대로 채워.
- slide_number는 원문에 표기된 번호를 그대로 사용해. (원문에 1, 3, 5만 있으면 그대로 1, 3, 5로 채워.)
- script에는 구분 기호(예: "1.", "Slide 2:") 자체와 구분 기호 뒤에 나오는 소제목은 제거하고, 그 슬라이드에 해당하는 본문 텍스트만 담아.
- keywords에는 해당 슬라이드 대본에서 추출한 중요한 단어 목록을 3~7개 정도 담아.

# status="fail"인 경우
- slide_number은 None으로 채우고, script에는 구분할 수 없는 전체 대본을 그대로 담아.
- keywords는 15개 정도 추출해서 담아.
- 키워드가 등장하는 구간은 편향되지 않게 해줘. (대본의 앞쪽에만 키워드가 몰리지 않도록)
"""


#### LLM 모델 불러오기

In [70]:
os.getenv('OPENAI_MODEL'), os.getenv('OPENAI_BASE_URL')

('openai/gpt-5.6-luna',
 'https://mlapi.run/286e9158-d32e-436d-a23d-36b43fc8e68a/v1')

In [71]:
# LLM 객체 생성
model = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
)

# Structured Output 연결
structured_output = model.with_structured_output(SeperatedSlides)

# 실제 모델 호출이 되는지 간단한 테스트
response = structured_output.invoke("Please separate the following script into slides:\n\n안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.") 
response

SeperatedSlides(status='success', slides=[Slide(slide_number=1, script='안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.', keywords=['멸종위기청년', '발표'], highlights=['멸종위기청년'])], terms=['멸종위기청년'])

#### 테스트용 대본 파일 import

In [72]:
scripts = []

# 테스트용 .txt 파일 읽기
path = os.path.join(os.getcwd(), "대본")
for filename in os.listdir(path):
    if filename.endswith(".txt"):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            script = file.read()
            scripts.append(script)
            
scripts

['슬라이드 1:\n\n안녕하세요. 오늘은 선형대수학에서 다루는 고유값과 고유벡터의 개념에 대해 발표하겠습니다.\n\n이 개념은 정방행렬의 성질을 이해하는 데 핵심적인 역할을 하며, 이후 배우게 될 특이값 분해와도 밀접하게 연결됩니다.\n\n슬라이드 2:\n\n행렬 A에 대해 Ax = λx를 만족하는 0이 아닌 벡터 x가 존재할 때, λ를 고유값, x를 고유벡터라고 부릅니다.\n\n고유값은 특성방정식인 det(A - λI) = 0을 풀어서 구할 수 있으며, 이 방정식의 해의 개수는 행렬의 차수와 같습니다.\n\n슬라이드 3:\n\n고유값 분해를 이용하면 대각화가 가능한 행렬을 A = PDP⁻¹ 형태로 표현할 수 있습니다.\n\n여기서 D는 고유값으로 이루어진 대각행렬이고, P는 고유벡터를 열벡터로 갖는 행렬입니다.\n\n모든 행렬이 대각화 가능한 것은 아니며, 고유벡터가 선형독립이 아닌 경우에는 조르당 표준형을 사용해야 합니다.\n\n슬라이드 4:\n\n대칭행렬의 경우에는 스펙트럼 정리에 의해 항상 직교대각화가 가능합니다.\n\n이 성질은 주성분분석, 즉 PCA와 같은 차원 축소 기법에서 공분산 행렬을 다룰 때 중요하게 활용됩니다.\n\n슬라이드 5:\n\n정방행렬이 아닌 일반적인 행렬에도 적용할 수 있는 것이 특이값 분해, SVD입니다.\n\n행렬 A는 A = UΣVᵀ로 분해되며, U와 V는 직교행렬, Σ는 특이값을 대각선에 가진 행렬입니다.\n\n슬라이드 6:\n\n고유값과 특이값 분해는 추천 시스템, 이미지 압축, 자연어처리의 잠재의미분석 등 다양한 분야에 응용됩니다.\n\n오늘 다룬 개념들이 이후 수치해석이나 머신러닝 과목을 학습하실 때 중요한 기반이 되길 바랍니다.\n\n감사합니다.',
 '# 분산 데이터베이스의 일관성과 합의 알고리즘\n\n## 01. 문제 상황\n\n안녕하세요. 저희는 분산 데이터베이스에서 발생하는 일관성 문제와 이를 해결하기 위한 합의 알고리즘을 소개합니다.\n\n여러 노드에 데이터를 복제해 저장할 때 네트워크 지연이나 장애가 발

#### 전처리 함수 정의 (LLM 응답 후처리용)

In [73]:
import re

def clean_script(text: str) -> str:
    # 헤더(#, ##, ...) 제거
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    # 굵게/기울임 강조 기호 제거: **text**, __text__, *text*, _text_
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    text = re.sub(r"__(.+?)__", r"\1", text)
    text = re.sub(r"(?<!\w)\*(.+?)\*(?!\w)", r"\1", text)
    text = re.sub(r"(?<!\w)_(.+?)_(?!\w)", r"\1", text)
    # 인라인 코드 백틱(`code`) 제거
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # 마크다운 링크 [텍스트](URL) -> 텍스트 (슬라이드 마커 [슬라이드 1] 등은 뒤에 괄호가 없어 영향 없음)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # 개행 및 연속 공백을 단일 공백으로 정리
    text = re.sub(r"\s+", " ", text)
    return text.strip()


#### 단일 대본 test

In [74]:
result = structured_output.invoke(
    [
        ("system", SYSTEM_PROMPT),
        ("user", scripts[2])
    ]
)
data = result.model_dump()
data


{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '최근 자연어처리 분야에서는 트랜스포머 아키텍처를 기반으로 한 대규모 언어모델이 핵심 기술로 자리잡았습니다.\n\n그렇다면 이러한 모델은 어떻게 방대한 텍스트로부터 언어의 패턴을 학습할 수 있을까요?\n\n오늘 발표에서는 어텐션 메커니즘과 사전학습 과정을 중심으로 살펴보겠습니다.',
   'keywords': ['자연어처리',
    '트랜스포머 아키텍처',
    '대규모 언어모델',
    '방대한 텍스트',
    '어텐션 메커니즘',
    '사전학습'],
   'highlights': []},
  {'slide_number': 2,
   'script': '트랜스포머의 핵심은 셀프 어텐션 메커니즘입니다.\n\n입력 시퀀스의 각 토큰은 쿼리, 키, 밸류 벡터로 변환되고, 쿼리와 키의 내적을 통해 다른 토큰과의 연관성을 계산합니다.\n\n이 과정을 통해 모델은 문장 내에서 멀리 떨어진 토큰 간의 의존관계도 포착할 수 있습니다.',
   'keywords': ['셀프 어텐션 메커니즘', '입력 시퀀스', '쿼리', '밸류 벡터', '연관성', '의존관계'],
   'highlights': []},
  {'slide_number': 3,
   'script': '언어모델은 마스크 언어모델링이나 다음 토큰 예측과 같은 자기지도학습 방식으로 사전학습됩니다.\n\n이후 특정 과제에 맞게 소량의 데이터로 파인튜닝하거나, 인간 피드백 기반 강화학습인 RLHF를 적용해 응답 품질을 개선합니다.',
   'keywords': ['마스크 언어모델링',
    '다음 토큰 예측',
    '자기지도학습',
    '사전학습',
    '파인튜닝',
    'RLHF',
    '응답 품질'],
   'highlights': []},
  {'slide_number': 4,
   'script': '파라미터를 갱신하지 않고도 프롬프트에 제시된 예시만으로 새로운

In [75]:
for slide in data["slides"]:
    slide["script"] = clean_script(slide["script"])
data

{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '최근 자연어처리 분야에서는 트랜스포머 아키텍처를 기반으로 한 대규모 언어모델이 핵심 기술로 자리잡았습니다. 그렇다면 이러한 모델은 어떻게 방대한 텍스트로부터 언어의 패턴을 학습할 수 있을까요? 오늘 발표에서는 어텐션 메커니즘과 사전학습 과정을 중심으로 살펴보겠습니다.',
   'keywords': ['자연어처리',
    '트랜스포머 아키텍처',
    '대규모 언어모델',
    '방대한 텍스트',
    '어텐션 메커니즘',
    '사전학습'],
   'highlights': []},
  {'slide_number': 2,
   'script': '트랜스포머의 핵심은 셀프 어텐션 메커니즘입니다. 입력 시퀀스의 각 토큰은 쿼리, 키, 밸류 벡터로 변환되고, 쿼리와 키의 내적을 통해 다른 토큰과의 연관성을 계산합니다. 이 과정을 통해 모델은 문장 내에서 멀리 떨어진 토큰 간의 의존관계도 포착할 수 있습니다.',
   'keywords': ['셀프 어텐션 메커니즘', '입력 시퀀스', '쿼리', '밸류 벡터', '연관성', '의존관계'],
   'highlights': []},
  {'slide_number': 3,
   'script': '언어모델은 마스크 언어모델링이나 다음 토큰 예측과 같은 자기지도학습 방식으로 사전학습됩니다. 이후 특정 과제에 맞게 소량의 데이터로 파인튜닝하거나, 인간 피드백 기반 강화학습인 RLHF를 적용해 응답 품질을 개선합니다.',
   'keywords': ['마스크 언어모델링',
    '다음 토큰 예측',
    '자기지도학습',
    '사전학습',
    '파인튜닝',
    'RLHF',
    '응답 품질'],
   'highlights': []},
  {'slide_number': 4,
   'script': '파라미터를 갱신하지 않고도 프롬프트에 제시된 예시만으로 새로운 과제를 수행하는 능력을 인

#### highlight 구간 추출

In [76]:
for slide in data["slides"]:
    for keyword in slide["keywords"]:
        if keyword not in slide["script"]:
            print(f"Keyword '{keyword}' not found in slide {slide['slide_number']}.")
        start = slide["script"].find(keyword)
        end = start + len(keyword)
        slide["highlights"].append(f"{start}:{end}")
data

{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '최근 자연어처리 분야에서는 트랜스포머 아키텍처를 기반으로 한 대규모 언어모델이 핵심 기술로 자리잡았습니다. 그렇다면 이러한 모델은 어떻게 방대한 텍스트로부터 언어의 패턴을 학습할 수 있을까요? 오늘 발표에서는 어텐션 메커니즘과 사전학습 과정을 중심으로 살펴보겠습니다.',
   'keywords': ['자연어처리',
    '트랜스포머 아키텍처',
    '대규모 언어모델',
    '방대한 텍스트',
    '어텐션 메커니즘',
    '사전학습'],
   'highlights': ['3:8', '15:25', '34:42', '77:84', '117:125', '127:131']},
  {'slide_number': 2,
   'script': '트랜스포머의 핵심은 셀프 어텐션 메커니즘입니다. 입력 시퀀스의 각 토큰은 쿼리, 키, 밸류 벡터로 변환되고, 쿼리와 키의 내적을 통해 다른 토큰과의 연관성을 계산합니다. 이 과정을 통해 모델은 문장 내에서 멀리 떨어진 토큰 간의 의존관계도 포착할 수 있습니다.',
   'keywords': ['셀프 어텐션 메커니즘', '입력 시퀀스', '쿼리', '밸류 벡터', '연관성', '의존관계'],
   'highlights': ['11:22', '27:33', '41:43', '48:53', '83:86', '128:132']},
  {'slide_number': 3,
   'script': '언어모델은 마스크 언어모델링이나 다음 토큰 예측과 같은 자기지도학습 방식으로 사전학습됩니다. 이후 특정 과제에 맞게 소량의 데이터로 파인튜닝하거나, 인간 피드백 기반 강화학습인 RLHF를 적용해 응답 품질을 개선합니다.',
   'keywords': ['마스크 언어모델링',
    '다음 토큰 예측',
    '자기지도학습',
    '사전학습',
    '파인튜닝',
    'RLHF',
    '응답 품질'],

#### 전체 대본 test

In [77]:
# results = []

# for script in scripts:
#     result = structured_output.invoke(
#         [
#             ("system", SYSTEM_PROMPT),
#             ("user", script)
#         ]
#     )
#     data = result.model_dump()
#     for slide in data["slides"]:
#         slide["script"] = clean_script(slide["script"])
#     results.append(data)

# results
